# 03: Digital Persona Twin (DPT) & Evolution Model (DToC)
**AI Shopping OS — Memory-Augmented Multi-Agent AI Shopping Assistant**

This notebook builds:
1. **DPT Cold-Start Persona Clustering:** K-Means clustering on 9 onboarding traits into 6 target customer archetypes.
2. **DToC Dynamic Trajectory Evolution:** A **PyTorch LSTM model** predicting behavioral trait deltas ($\Delta T$) from sequences of user shopping interaction signals.


In [ ]:
# Step 1: Install Dependencies
!pip install -q scikit-learn matplotlib seaborn pandas numpy torch joblib


In [ ]:
# Step 2: Generate Persona Trait Vectors for Clustering
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

np.random.seed(42)
N_USERS = 1200

TRAITS = [
    "budgetFlexibility", "brandLoyalty", "priceSensitivity",
    "purchaseConfidence", "technologyAdoption", "impulseBuyingScore",
    "discountPreference", "comparisonFrequency", "shoppingConsistency"
]

# Generate synthetic 9-axis trait vectors (0..100)
traits_matrix = np.random.randint(10, 95, size=(N_USERS, 9))
df_traits = pd.DataFrame(traits_matrix, columns=TRAITS)

# Fit K-Means with k=6
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_traits['cluster'] = kmeans.fit_predict(df_traits[TRAITS])

CLUSTER_NAMES = {
    0: "Budget-Conscious Shopper",
    1: "Premium Buyer",
    2: "Student Shopper",
    3: "Fashion Enthusiast",
    4: "Family Shopper",
    5: "Tech Enthusiast"
}
df_traits['persona_name'] = df_traits['cluster'].map(CLUSTER_NAMES)

print("Cluster Distribution:")
print(df_traits['persona_name'].value_counts())


In [ ]:
# Step 3: Visualize Persona Clusters with PCA (2D Projection)
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(df_traits[TRAITS])

plt.figure(figsize=(10, 6))
for cluster_id in range(6):
    mask = df_traits['cluster'] == cluster_id
    plt.scatter(coords_2d[mask, 0], coords_2d[mask, 1], label=CLUSTER_NAMES[cluster_id], alpha=0.7)

plt.title("DPT Persona Cluster Map (PCA Reduced)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig("persona_clusters_pca.png")
plt.show()


In [ ]:
# Step 4: DToC Evolution Trajectory Model (PyTorch LSTM)
import torch
import torch.nn as nn
import torch.optim as optim

class TwinEvolutionLSTM(nn.Module):
    def __init__(self, signal_dim=8, hidden_dim=32, trait_dim=9):
        super(TwinEvolutionLSTM, self).__init__()
        self.lstm = nn.LSTM(signal_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, trait_dim)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, signal_dim)
        lstm_out, (hn, cn) = self.lstm(x)
        delta_traits = self.fc(hn[-1])
        return delta_traits

# Instantiate model
signal_dim = 8 # [search, compare, purchase, wishlist, price_check, review_read, filter_budget, cart_add]
model = TwinEvolutionLSTM(signal_dim=signal_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# Dummy training pass demo
dummy_sequences = torch.randn(32, 5, signal_dim) # 32 users, 5 interaction sequence steps
dummy_targets = torch.randn(32, 9) * 2.0 # trait delta updates

optimizer.zero_grad()
outputs = model(dummy_sequences)
loss = criterion(outputs, dummy_targets)
loss.backward()
optimizer.step()

print(f"Twin Trajectory Evolution Model initialized. Sample Step Loss: {loss.item():.4f}")


In [ ]:
# Step 5: Export Models
import joblib

joblib.dump(kmeans, "dpt_kmeans_persona.joblib")
torch.save(model.state_dict(), "dtoc_evolution_lstm.pt")
print("Saved DPT K-Means centroids and PyTorch DToC state dict.")
